In [1]:
!pip install --upgrade gradio gradio_client

  Using cached gradio-6.15.2-py3-none-any.whl.metadata (17 kB)
Using cached gradio-6.15.2-py3-none-any.whl (20.1 MB)


In [5]:
import os, gradio as gr, pypdf
from groq import Groq

client = Groq(api_key="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")  # ← your key
pdfs = {}

def upload(files):
    for f in files or []:
        reader = pypdf.PdfReader(f)
        pdfs[os.path.basename(f)] = "\n".join(p.extract_text() for p in reader.pages)
    return f"✅ {len(pdfs)} PDF(s): {', '.join(pdfs.keys())}"

def chat(q, history):
    if not pdfs: return history + [{"role":"user","content":q}, {"role":"assistant","content":"⚠️ Upload a PDF first."}] , ""
    text = "\n".join(f"[{n}]\n{t[:4000]}" for n,t in pdfs.items())
    r = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role":"system","content":f"Answer from PDFs only:\n{text}"},{"role":"user","content":q}],
        max_tokens=512)
    return history + [{"role":"user","content":q}, {"role":"assistant","content":r.choices[0].message.content}], ""

with gr.Blocks() as app:
    gr.Markdown("# 📄 PDF Chatbot")
    f = gr.File(file_types=[".pdf"], file_count="multiple", label="Upload PDFs")
    s = gr.Textbox(label="Status", interactive=False)
    c = gr.Chatbot()
    m = gr.Textbox(placeholder="Ask about your PDFs...")
    f.change(upload, f, s)
    m.submit(chat, [m, c], [c, m])

app.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://8e714c5cb945ffcf4d.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
